In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from uncertainties import ufloat
import uncertainties.unumpy as unp

In [ ]:
#g2 = np.loadtxt("AAA_g_3x4.txt")**2
g2 = 100
Eground, Estring = [], []
Nx = 4
Ny = np.array([*range(3, 8)])
r = Ny
for Ny_ in Ny:
    Eground.append(float(np.loadtxt(f"ground_state/g2_100.00/AAA_E_4x{Ny_}.txt")))
    Estring.append(float(np.loadtxt(f"string_state/g2_100.00/AAA_E_4x{Ny_}.txt")))
print(Eground)
print(Estring)
V = np.array(Estring) - np.array(Eground)
print(V)

def cornell(r, gamma, sigma, c):
    return -gamma / r + sigma * r + c

def simplified(r, sigma, c):
    return sigma * r + c

opt, cov = sp.optimize.curve_fit(cornell, xdata=r, ydata=V)
gamma, sigma, c = unp.uarray(opt, np.diag(cov)**0.5)
print(f"gamma={gamma}, sigma={sigma}, c={c}")

gamma_theo = 8 * np.pi / (3 * g2)
sigma_theo = 3 * g2 - 64 / (np.pi * g2)
c_theo = (32 - 64 / np.pi) / g2
print(f"Theoretical: gamma={gamma_theo}, sigma={sigma_theo}, c={c_theo}")

r_fit = np.linspace(2, 8, 100)
V_fit = cornell(r_fit, *opt)

# chi2 = (sum (V - cornell(r, *opt))**2/V**2 / dV_rel**2) / (len(r) - len(opt)) = 1
# => (len(r) - len(opt)) * dV_rel**2 = sum (V - cornell(r, *opt))**2/V**2
# => dV_rel = (sum (V - cornell(r, *opt))**2/V**2 / (len(r) - len(opt)))**0.5
dV_rel = (np.sum((V - cornell(r, *opt))**2/V**2) / (len(r) - len(opt)))**0.5
chi2 = np.sum((V - cornell(r, *opt))**2/V**2) / dV_rel**2 / (len(r) - len(opt))
print(f"dV={round(dV_rel*100,2)}%, chi2={chi2}")

plt.figure()
plt.xlabel(r"$N_y$")
plt.ylabel(r"$V$")
plt.grid(which="both")
plt.plot(r_fit, V_fit)
plt.errorbar(r, V, V*dV_rel, fmt='o', ls='', capsize=5)